# Module 03: Matplotlib for Machine Learning
## Notebook 03: Multi-Plot Layouts and Subplots

In real-world data science, single isolated plots cannot convey the comprehensive state of a machine learning experiment. You need multi-panel diagnostic dashboards: comparing feature distributions across classes, tracking multiple loss components, or auditing residual error behavior.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Create multi-panel figures using `plt.subplots(nrows, ncols)`.
2. Programmatically iterate through subplots using `.ravel()`.
3. Synchronize axes across plots using `sharex=True` and `sharey=True`.
4. Construct asymmetric layouts with varying row/column spans using `GridSpec`.
5. Adjust spacing and padding cleanly using `plt.tight_layout()` and `subplots_adjust`.
6. **Advanced:** Construct **Inset Zoom Subplots (`ax.inset_axes`)** to magnify transient anomalies or convergence details within a parent chart.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
print("Matplotlib loaded!")

### 1. Creating Grids with `plt.subplots()`

`fig, axes = plt.subplots(nrows, ncols)` returns:
- `fig`: The parent `Figure`.
- `axes`: A 1D or 2D NumPy array of `Axes` objects!
- Using `axes.ravel()` flattens the array so you can iterate cleanly in a single `for` loop.

In [ ]:
# Generating 4 feature distributions
feature_names = ['Account_Tenure', 'Monthly_Usage', 'Customer_Age', 'Support_Calls']
features_data = [
    rng.exponential(scale=15, size=400),
    rng.normal(loc=120, scale=30, size=400),
    rng.integers(18, 70, size=400),
    rng.poisson(lam=2.5, size=400)
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharey=False)
axes_flat = axes.ravel()  # Flatten (2, 2) array into 1D of 4 elements

colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b3']

for i in range(len(features_data)):
    ax = axes_flat[i]
    ax.hist(features_data[i], bins=20, color=colors[i], edgecolor='black', alpha=0.7)
    ax.set_title(f"Distribution: {feature_names[i]}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
    ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

---
### 2. Synchronizing Axes with `sharex` and `sharey`

When comparing identical metrics across different models or splits, keeping axis limits unlinked confuses interpretation. Setting `sharey=True` ensures the scale is identical across panels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

models = ['Logistic Regression', 'Random Forest', 'Gradient Boosting']
accuracies = [rng.normal(0.82, 0.03, 50), rng.normal(0.89, 0.02, 50), rng.normal(0.92, 0.015, 50)]

for ax, model_name, acc in zip(axes, models, accuracies):
    ax.boxplot(acc, tick_labels=[model_name], patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='blue'))
    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.6)

axes[0].set_ylabel("Cross-Validation Accuracy", fontsize=11)
plt.tight_layout()
plt.show()

---
### 3. Asymmetric Layouts with `GridSpec`

In many diagnostic dashboards, you want one primary prominent chart (e.g. training loss covering the entire top row) and smaller secondary plots underneath. `plt.GridSpec` provides full control over row and column spans!

In [ ]:
fig = plt.figure(figsize=(11, 7))
gs = fig.add_gridspec(2, 2, height_ratios=[1.2, 1])

# Large plot spanning the entire top row
ax_top = fig.add_subplot(gs[0, :])

# Two smaller plots on the bottom row
ax_bottom_left = fig.add_subplot(gs[1, 0])
ax_bottom_right = fig.add_subplot(gs[1, 1])

# Plot data
epochs = np.arange(1, 31)
loss = 1.8 * np.exp(-0.15 * epochs) + 0.1
ax_top.plot(epochs, loss, 'o-', color='navy', linewidth=2)
ax_top.set_title("Primary Diagnostic: Global Optimization Loss Curve", fontsize=13, fontweight='bold')
ax_top.set_xlabel("Epoch")
ax_top.set_ylabel("Loss")
ax_top.grid(True, linestyle='--', alpha=0.5)

# Bottom left: Learning rate decay schedule
lr_schedule = 0.01 * (0.95 ** epochs)
ax_bottom_left.plot(epochs, lr_schedule, color='crimson', linewidth=2)
ax_bottom_left.set_title("Learning Rate Schedule (Exponential Decay)", fontsize=11)
ax_bottom_left.set_xlabel("Epoch")
ax_bottom_left.set_ylabel("Learning Rate")
ax_bottom_left.grid(True, linestyle=':', alpha=0.5)

# Bottom right: Gradient Norms
grad_norms = 0.8 * np.exp(-0.12 * epochs) + rng.normal(0, 0.03, size=len(epochs))
ax_bottom_right.plot(epochs, grad_norms, color='darkgreen', linewidth=1.5)
ax_bottom_right.set_title("Gradient Norms ||g||", fontsize=11)
ax_bottom_right.set_xlabel("Epoch")
ax_bottom_right.set_ylabel("L2 Norm")
ax_bottom_right.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

---
### 4. Advanced Complex Usage: Inset Zoom Subplots (`ax.inset_axes`) for Anomaly Inspection

In high-resolution loss curves or time series signals:
- Long runs (100+ epochs) obscure micro-scale phenomena (e.g. transient gradient oscillations, loss spikes during learning rate warmups, or fine-grained convergence).
- **`ax.inset_axes([x0, y0, width, height])`**: Creates an embedded miniature coordinate plane inside the parent Axes.
- **`ax.indicate_inset_zoom()`**: Automatically renders connecting bounding box lines from the region of interest to the inset!

In [ ]:
# Simulate a 100-epoch training curve with fine-grained oscillations around convergence
epochs_100 = np.arange(1, 101)
base_loss = 2.0 * np.exp(-0.06 * epochs_100) + 0.15
# Add a micro-spike disturbance between epochs 70 and 85
disturbances = np.sin(epochs_100 * 0.8) * 0.015
disturbances[70:85] += 0.04 * np.sin(np.linspace(0, np.pi, 15))
full_loss = base_loss + disturbances

fig, ax_main = plt.subplots(figsize=(10, 5.5))

# Plot full run on primary axes
ax_main.plot(epochs_100, full_loss, color='#2c3e50', linewidth=2, label='Training Loss')
ax_main.set_title("100-Epoch Optimization with Inset Zoom on Convergence Region", fontsize=13, fontweight='bold')
ax_main.set_xlabel("Epoch", fontsize=11)
ax_main.set_ylabel("Objective Loss", fontsize=11)
ax_main.grid(True, linestyle='--', alpha=0.5)

# Create Inset Axes: [x_start, y_start, width, height] in normalized (0 to 1) parent coordinates
ax_inset = ax_main.inset_axes([0.45, 0.40, 0.50, 0.52])

# Plot identical data inside the inset
ax_inset.plot(epochs_100, full_loss, 'o-', color='#e74c3c', markersize=3, linewidth=1.5)
# Set zoom limits to focus specifically on the epoch 65 to 90 regime
ax_inset.set_xlim(65, 90)
ax_inset.set_ylim(0.14, 0.26)
ax_inset.set_title("Magnified Convergence Phase (Epochs 65-90)", fontsize=9, fontweight='bold')
ax_inset.grid(True, linestyle=':', alpha=0.6)

# Connect inset to parent coordinate bounds
ax_main.indicate_inset_zoom(ax_inset, edgecolor='black', alpha=0.7)

ax_main.legend(loc='lower left')
plt.show()

### Summary & Next Steps
In this notebook, you mastered:
- Subplot grids and axis sharing (`sharex`, `sharey`).
- Programmatic subplot population with `.ravel()`.
- Advanced multi-span layout architecture with `GridSpec`.
- Inset zoom subplots (`ax.inset_axes` and `indicate_inset_zoom`) for micro-structural inspection.

**Next Notebook:** `04_advanced_customization_and_ml_visualizations.ipynb` — Decision boundary contouring, multi-class ROC curves, and 3D optimization surfaces.